# One historical NSE week — all requested timeframes

This focused notebook downloads the open **BANKNIFTY one-minute dataset**, selects the first trading week of 2010, creates every requested timeframe, validates the bars, visualizes them, and exports them to CSV.

Why BANKNIFTY instead of HDFCBANK: I could not verify a genuinely open individual-stock one-minute source covering January 2010. The open Kaggle index dataset explicitly covers BANKNIFTY from 2010–2020. Do not label index results as HDFCBANK results.

Sources:
- Dataset: https://www.kaggle.com/datasets/nishanthsalian/indian-stock-index-1minute-data-2008-2020
- NSE timing announcement: https://nsearchives.nseindia.com/content/press/17122009.htm

January 1, 2010 was an exchange holiday. The selected week is therefore **January 4–8, 2010**. The normal market opened at **09:00 IST** from January 4, 2010; using today's 09:15 anchor would be historically incorrect.

## Cell 1 — Install libraries

In [ ]:
%pip install -q kagglehub pandas numpy plotly pyarrow

## Cell 2 — Imports and exact historical configuration

In [ ]:
from pathlib import Path
import re
import warnings

import kagglehub
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)

INSTRUMENT = 'BANKNIFTY'
WEEK_START = pd.Timestamp('2010-01-04', tz='Asia/Kolkata')
WEEK_END_EXCLUSIVE = pd.Timestamp('2010-01-09', tz='Asia/Kolkata')
SELECTED_DAY = '2010-01-04'
NSE_TIMEZONE = 'Asia/Kolkata'
HISTORICAL_SESSION_OPEN = '09:00'
HISTORICAL_SESSION_LAST_MINUTE = '15:29'

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'research':
    PROJECT_ROOT = PROJECT_ROOT.parent
OUTPUT_DIR = PROJECT_ROOT / 'research' / 'data' / 'banknifty_2010_week'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Instrument: {INSTRUMENT}')
print(f'Week: {WEEK_START.date()} through {(WEEK_END_EXCLUSIVE - pd.Timedelta(days=1)).date()}')
print(f'Historical session anchor: {HISTORICAL_SESSION_OPEN} IST')

## Cell 3 — Download the open one-minute dataset

KaggleHub downloads and caches the public dataset. Its dataset license is CC BY-NC-SA 4.0; check the data card before redistributing derived files.

In [ ]:
DATASET_SLUG = 'nishanthsalian/indian-stock-index-1minute-data-2008-2020'
dataset_root = Path(kagglehub.dataset_download(DATASET_SLUG))

print(f'Dataset cached at: {dataset_root}')
all_files = [p for p in dataset_root.rglob('*') if p.is_file()]
print(f'Files found: {len(all_files):,}')
display(pd.DataFrame({'file': [str(p.relative_to(dataset_root)) for p in all_files[:50]]}))

## Cell 4 — Discover and parse BANKNIFTY files robustly

Community datasets often change folder names and CSV column casing. This parser discovers likely files, normalizes their schema, and retains only genuine timestamps in the requested week.

In [ ]:
def normalized_name(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())

def find_column(columns, aliases):
    mapping = {normalized_name(c): c for c in columns}
    for alias in aliases:
        if normalized_name(alias) in mapping:
            return mapping[normalized_name(alias)]
    return None

def parse_datetime_columns(frame):
    date_col = find_column(frame.columns, ['date', 'tradingdate'])
    time_col = find_column(frame.columns, ['time', 'tradingtime'])
    datetime_col = find_column(frame.columns, ['datetime', 'timestamp', 'date_time'])

    if datetime_col is not None:
        text = frame[datetime_col].astype(str).str.strip()
    elif date_col is not None and time_col is not None:
        text = frame[date_col].astype(str).str.strip() + ' ' + frame[time_col].astype(str).str.strip()
    elif date_col is not None:
        text = frame[date_col].astype(str).str.strip()
    else:
        text = frame.iloc[:, 0].astype(str).str.strip()

    # Handles YYYYMMDD HH:MM, ISO dates, and common day-first community files.
    parsed = pd.to_datetime(text, errors='coerce', format='mixed', dayfirst=True)
    return parsed

def parse_market_file(path):
    attempts = []
    for kwargs in ({}, {'sep': ';'}, {'header': None}):
        try:
            frame = pd.read_csv(path, low_memory=False, **kwargs)
            if frame.shape[1] >= 5:
                attempts.append(frame)
        except Exception:
            pass

    for frame in attempts:
        # Headerless fallback: date, time, open, high, low, close, optional volume.
        if all(isinstance(c, int) for c in frame.columns):
            names = ['date', 'time', 'open', 'high', 'low', 'close', 'volume'][:frame.shape[1]]
            frame = frame.iloc[:, :len(names)].copy()
            frame.columns = names

        open_col = find_column(frame.columns, ['open', 'o'])
        high_col = find_column(frame.columns, ['high', 'h'])
        low_col = find_column(frame.columns, ['low', 'l'])
        close_col = find_column(frame.columns, ['close', 'c', 'last'])
        volume_col = find_column(frame.columns, ['volume', 'vol', 'v'])
        if None in (open_col, high_col, low_col, close_col):
            continue

        timestamp = parse_datetime_columns(frame)
        out = pd.DataFrame({
            'timestamp': timestamp,
            'open': pd.to_numeric(frame[open_col], errors='coerce'),
            'high': pd.to_numeric(frame[high_col], errors='coerce'),
            'low': pd.to_numeric(frame[low_col], errors='coerce'),
            'close': pd.to_numeric(frame[close_col], errors='coerce'),
            'volume': (
                pd.to_numeric(frame[volume_col], errors='coerce')
                if volume_col is not None else np.nan
            ),
        }).dropna(subset=['timestamp', 'open', 'high', 'low', 'close'])
        if not out.empty:
            return out
    return pd.DataFrame()

candidate_files = [
    p for p in all_files
    if p.suffix.lower() in {'.csv', '.txt'}
    and ('banknifty' in normalized_name(p) or 'banknifty' in normalized_name(p.parent))
]
if not candidate_files:
    candidate_files = [p for p in all_files if p.suffix.lower() in {'.csv', '.txt'}]

week_parts = []
source_rows = []
for path in candidate_files:
    parsed = parse_market_file(path)
    if parsed.empty:
        continue
    if parsed['timestamp'].dt.tz is None:
        parsed['timestamp'] = parsed['timestamp'].dt.tz_localize(NSE_TIMEZONE)
    else:
        parsed['timestamp'] = parsed['timestamp'].dt.tz_convert(NSE_TIMEZONE)
    selected = parsed[
        (parsed.timestamp >= WEEK_START) &
        (parsed.timestamp < WEEK_END_EXCLUSIVE)
    ].copy()
    if not selected.empty:
        selected['source_file'] = str(path.relative_to(dataset_root))
        week_parts.append(selected)
        source_rows.append((str(path.relative_to(dataset_root)), len(selected)))

if not week_parts:
    raise RuntimeError('No BANKNIFTY one-minute rows were found for 2010-01-04 through 2010-01-08.')

raw_week = pd.concat(week_parts, ignore_index=True)
raw_week = raw_week.sort_values('timestamp').drop_duplicates('timestamp', keep='last')
raw_week = raw_week.set_index('timestamp')

print(f'Parsed one-minute rows: {len(raw_week):,}')
display(pd.DataFrame(source_rows, columns=['source_file', 'selected_rows']))
display(raw_week.head())
display(raw_week.tail())

## Cell 5 — Validate the raw historical week

Rows outside the historically correct 09:00–15:29 continuous session are removed. Data gaps remain visible; the notebook does not forward-fill OHLC prices.

In [ ]:
one_minute = raw_week.between_time(
    HISTORICAL_SESSION_OPEN, HISTORICAL_SESSION_LAST_MINUTE, inclusive='both'
).copy()
one_minute = one_minute[one_minute.index.dayofweek < 5]

ohlc_valid = (
    (one_minute.high >= one_minute[['open', 'close']].max(axis=1)) &
    (one_minute.low <= one_minute[['open', 'close']].min(axis=1)) &
    (one_minute.high >= one_minute.low) &
    (one_minute[['open', 'high', 'low', 'close']] > 0).all(axis=1)
)

quality_rows = []
for session_date, session in one_minute.groupby(one_minute.index.date):
    ordered = session.sort_index()
    minute_steps = ordered.index.to_series().diff().dt.total_seconds().div(60)
    quality_rows.append({
        'date': session_date,
        'rows': len(ordered),
        'first': ordered.index.min().strftime('%H:%M'),
        'last': ordered.index.max().strftime('%H:%M'),
        'missing_minute_gaps': int((minute_steps.dropna() > 1).sum()),
        'invalid_ohlc_rows': int((~ohlc_valid.loc[ordered.index]).sum()),
        'volume_available': bool(ordered.volume.notna().any()),
    })

quality_report = pd.DataFrame(quality_rows)
display(quality_report)
assert ohlc_valid.all(), 'Invalid OHLC rows detected; inspect before resampling.'

## Cell 6 — Create all nine requested timeframes

Every intraday bar is anchored to the historical 09:00 session open and is prevented from crossing into another trading day. Partial final bars are retained and labelled.

In [ ]:
AGGREGATION = {
    'open': 'first', 'high': 'max', 'low': 'min',
    'close': 'last', 'volume': 'sum'
}
RULES = {
    '1_minute': None,
    '5_minutes': '5min',
    '10_minutes': '10min',
    '15_minutes': '15min',
    '30_minutes': '30min',
    '60_minutes': '60min',
    '2_hours': '120min',
    '5_hours': '300min',
}
EXPECTED_SOURCE_MINUTES = {
    '1_minute': 1, '5_minutes': 5, '10_minutes': 10,
    '15_minutes': 15, '30_minutes': 30, '60_minutes': 60,
    '2_hours': 120, '5_hours': 300,
}

def resample_each_session(frame, rule):
    if rule is None:
        out = frame[['open', 'high', 'low', 'close', 'volume']].copy()
        out['source_minutes'] = 1
        return out

    sessions = []
    for _, session in frame.groupby(frame.index.date):
        kwargs = dict(
            rule=rule, origin='start_day', offset='9h',
            label='left', closed='left'
        )
        bars = session.resample(**kwargs).agg(AGGREGATION)
        bars['source_minutes'] = session.close.resample(**kwargs).count()
        bars = bars.dropna(subset=['open', 'high', 'low', 'close'])
        sessions.append(bars)
    return pd.concat(sessions).sort_index()

frames = {name: resample_each_session(one_minute, rule) for name, rule in RULES.items()}

daily = one_minute.groupby(one_minute.index.normalize()).agg(AGGREGATION)
daily['source_minutes'] = one_minute.close.groupby(one_minute.index.normalize()).count()
frames['1_trading_day'] = daily

coverage_rows = []
for name, frame in frames.items():
    expected = EXPECTED_SOURCE_MINUTES.get(name)
    partial = int((frame.source_minutes < expected).sum()) if expected else 0
    coverage_rows.append({
        'timeframe': name,
        'bars': len(frame),
        'first_bar': frame.index.min(),
        'last_bar': frame.index.max(),
        'partial_bars': partial,
    })

coverage = pd.DataFrame(coverage_rows)
display(coverage)

## Cell 7 — Inspect every timeframe for one selected day

In [ ]:
selected_date = pd.Timestamp(SELECTED_DAY).date()

for name, frame in frames.items():
    day_frame = frame[frame.index.date == selected_date]
    print(f'\n{name}: {len(day_frame)} bars on {SELECTED_DAY}')
    display(day_frame)

## Cell 8 — Add simple research indicators and conditions

The condition is only a demonstration. A single week is insufficient to validate a swing-trading strategy.

In [ ]:
def add_research_fields(frame):
    out = frame.copy()
    out['ema9'] = out.close.ewm(span=9, adjust=False).mean()
    out['ema20'] = out.close.ewm(span=20, adjust=False).mean()
    delta = out.close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
    out['rsi14'] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    out['condition_pass'] = (
        (out.close > out.ema9) &
        (out.ema9 > out.ema20) &
        out.rsi14.between(50, 70)
    ).fillna(False)
    return out

frames = {name: add_research_fields(frame) for name, frame in frames.items()}

latest_status = pd.DataFrame([
    {
        'timeframe': name,
        'latest_bar': frame.index[-1],
        'close': frame.close.iloc[-1],
        'ema9': frame.ema9.iloc[-1],
        'ema20': frame.ema20.iloc[-1],
        'rsi14': frame.rsi14.iloc[-1],
        'condition': 'PASS' if frame.condition_pass.iloc[-1] else 'WAIT',
    }
    for name, frame in frames.items() if not frame.empty
])
display(latest_status)

## Cell 9 — Visualize any timeframe

In [ ]:
VIEW_TIMEFRAME = '15_minutes'  # Change to any key shown in coverage
VIEW_ONLY_SELECTED_DAY = False

def plot_timeframe(frame, timeframe):
    view = frame.copy()
    if VIEW_ONLY_SELECTED_DAY:
        view = view[view.index.date == pd.Timestamp(SELECTED_DAY).date()]
    if view.empty:
        print('No bars to plot.')
        return

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.75, 0.25], vertical_spacing=0.04
    )
    fig.add_trace(go.Candlestick(
        x=view.index, open=view.open, high=view.high,
        low=view.low, close=view.close, name='OHLC'
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=view.index, y=view.ema9, name='EMA 9',
        line=dict(width=1.3, color='#2563eb')
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=view.index, y=view.ema20, name='EMA 20',
        line=dict(width=1.3, color='#f59e0b')
    ), row=1, col=1)
    passed = view[view.condition_pass]
    fig.add_trace(go.Scatter(
        x=passed.index, y=passed.low,
        mode='markers', name='Condition pass',
        marker=dict(symbol='triangle-up', size=9, color='#16a34a')
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=view.index, y=view.rsi14, name='RSI 14',
        line=dict(width=1.3, color='#7c3aed')
    ), row=2, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='#dc2626', row=2, col=1)
    fig.add_hline(y=50, line_dash='dot', line_color='gray', row=2, col=1)
    fig.update_layout(
        title=f'{INSTRUMENT} — {timeframe} — first trading week of 2010',
        template='plotly_white', height=800,
        hovermode='x unified', xaxis_rangeslider_visible=False
    )
    fig.show()

plot_timeframe(frames[VIEW_TIMEFRAME], VIEW_TIMEFRAME)

## Cell 10 — Export every timeframe to CSV and Parquet

In [ ]:
for name, frame in frames.items():
    export = frame.reset_index()
    export.to_csv(OUTPUT_DIR / f'{INSTRUMENT}_{name}_2010-01-04_to_2010-01-08.csv', index=False)
    export.to_parquet(OUTPUT_DIR / f'{INSTRUMENT}_{name}_2010-01-04_to_2010-01-08.parquet', index=False)

quality_report.to_csv(OUTPUT_DIR / 'raw_data_quality_report.csv', index=False)
coverage.to_csv(OUTPUT_DIR / 'timeframe_coverage.csv', index=False)
latest_status.to_csv(OUTPUT_DIR / 'latest_condition_status.csv', index=False)

print(f'Exported {len(frames)} timeframes to: {OUTPUT_DIR.resolve()}')
display(pd.DataFrame({'output_file': sorted(p.name for p in OUTPUT_DIR.iterdir())}))